**Tweets Using BERT for Sentiment Analysis**


In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("Tweets.csv")
df.columns = df.columns.str.strip()

df = df[["selected_text", "sentiment"]].dropna()
df = df.sample(2000)

texts = df["selected_text"].astype(str).tolist()

label_map = {"negative": 0, "neutral": 1, "positive": 2}
labels = [label_map[x] for x in df["sentiment"]]

In [3]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

In [4]:
# Dataset
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = TweetDataset(encodings, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [5]:
# Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# Training
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

KeyboardInterrupt: 

In [ ]:
# Evaluation on same data (simple)
model.eval()

preds = []
true = []

with torch.no_grad():
    for batch in loader:
        labels_batch = batch["labels"]
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        preds.extend(predictions)
        true.extend(labels_batch.numpy())

print("Accuracy:", accuracy_score(true, preds))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true, preds)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["neg", "neu", "pos"],
    yticklabels=["neg", "neu", "pos"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Prediction function
label_names = {0: "negative", 1: "neutral", 2: "positive"}

def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    print("Text:", text)
    print("Sentiment:", label_names[pred])

In [ ]:
# Test
predict("I love this phone but battery is average")